# Full validation of the chosen fine-tuned Qwen3-4B (Milestone 3)

Runs the checkpoint chosen by the selection run (`checkpoint-0792`, `docs/milestone-3/VALIDATION_LOG.md`) **once** on the full validation splits, as `docs/milestone-2/EVAL_PROTOCOL.md`, Amendment 1, requires:
- tier 1: 550 primary cases (dataset v2, splits v1 `validation`);
- tier 2: 250 primary cases (dataset v3, splits v2 `validation`).

The result is for reporting and does not change the choice. **Validation data only; no test split is touched.**

For each tier, `python -m packages.support_evaluator.checkpoint_selection validate`:
1. checks the checkpoint's `state.json` against its logged sha256, its adapter files against `state.json`, and that it is the checkpoint named in the selection run's `choice.json` (itself checked against its logged sha256);
2. merges the adapter into the fp16 base and evaluates with the declared settings on **one visible T4**, one generation at a time;
3. runs the fp16 smoke test first (10 seeded validation cases);
4. then runs every case of the tier, and writes `validation-tier<N>.json`: decision accuracy, wrong interventions on healthy runs, the monitored rates, the protocol summary, the metadata and every output file's sha256.

**On Kaggle (T4 x2):** tier 1 runs on GPU 0 and tier 2 on GPU 1 at the same time, each process seeing exactly one T4. `schedule.jsonl` records which GPU ran which tier, and when. **On Colab:** one T4, tier 1 then tier 2, with outputs on Google Drive so a disconnected session resumes.

**Setup, Kaggle:**

- Accelerator *GPU T4 x2*, Internet *on*, Environment *Pin to original environment*.
- Inputs:
  - the evaluation bundle dataset;
  - the session 2 retry training notebook's output, which holds `checkpoint-0792`;
  - the checkpoint-selection notebook's output (`selection-run-1`), which holds `qlora-selection/choice.json`.
- Set `BUNDLE_COMMIT`, then *Save Version → Save & Run All*.

**Setup, Colab:** *Runtime → Change runtime type → T4 GPU*. On Google Drive, under `MyDrive/hetren-m3/`, put the bundle, `checkpoints/checkpoint-0792/` (`state.json` and `adapter/`) and `qlora-selection/choice.json`. Set `PLATFORM = "colab"` and `BUNDLE_COMMIT`, then *Run all*.

In [ ]:
PLATFORM = "kaggle"  # "kaggle" or "colab"
BUNDLE_COMMIT = ""  # the evaluation bundle's commit (full sha); the notebook stops if the bundle found is another
CHOSEN = 792
STATE_SHA256 = "d6230ff6e1780ac5286f0accc0a45298dd8db49fc11ed6fdb1330d98189276d5"  # checkpoint-0792 (validation log)
CHOICE_SHA256 = "d41a7ef595e8882ff0dbb367e62b03b87c8f1006bb60da09811fb19f11992332"  # selection-run-1 choice.json
TIERS = [1, 2]
DRIVE = "/content/drive/MyDrive/hetren-m3"  # Colab only

import glob, hashlib, json, os, shutil, subprocess, sys, tarfile, threading, time

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    INPUTS, WORK, OUT = DRIVE, "/content/hetren", f"{DRIVE}/qlora-validation"
else:
    INPUTS, WORK, OUT = "/kaggle/input", "/kaggle/working/hetren", "/kaggle/working/qlora-validation"


def earlier_session(path):  # a copy of the code inside an attached earlier notebook output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob(f"{INPUTS}/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob(f"{INPUTS}/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if BUNDLE_COMMIT and COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

os.chdir(WORK)

In [ ]:
# The evaluation libraries, as in the selection run; torch is the platform's.
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
GPUS = [line.split(",")[1].strip() for line in subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()]

if not GPUS or any("T4" not in name for name in GPUS):
    raise SystemExit(f"STOP: the GPUs are {GPUS}; the protocol evaluates on T4s")

print(len(GPUS), "T4s; each evaluation process sees one")

## The chosen checkpoint and the selection run's choice, checked by hash; the base model downloaded once

In [ ]:
def find(pattern, digest):
    found = sorted(glob.glob(f"{INPUTS}/**/{pattern}", recursive=True))
    matching = [p for p in found if hashlib.sha256(open(p, "rb").read()).hexdigest() == digest]

    if not matching:
        raise SystemExit(f"STOP: no {pattern} with sha256 {digest} among {found}")

    return matching[0]


CHECKPOINT = os.path.dirname(find(f"checkpoint-{CHOSEN:04d}/state.json", STATE_SHA256))
CHOICE = find("qlora-selection/choice.json", CHOICE_SHA256)

if json.load(open(CHOICE))["chosen"] != f"checkpoint-{CHOSEN:04d}":
    raise SystemExit(f"STOP: choice.json chose {json.load(open(CHOICE))['chosen']}, not checkpoint-{CHOSEN:04d}")

print("checkpoint:", CHECKPOINT)
print("choice:", CHOICE)

subprocess.run([sys.executable, "-c", "from huggingface_hub import snapshot_download; "
                "from packages.hetren_baselines.local_model import QWEN_MODEL, QWEN_REVISION; "
                "print(snapshot_download(QWEN_MODEL, revision=QWEN_REVISION))"], check=True)

## Validation run (about 1 hour with the tiers on two T4s; about 1.8 hours on one)

Logs: `qlora-validation/logs/tier<N>.log`.

In [ ]:
os.makedirs(f"{OUT}/logs", exist_ok=True)
pending = [t for t in TIERS if not os.path.exists(f"{OUT}/tier{t}/validation-tier{t}.json")]
EXITS = {}
SCHEDULE = f"{OUT}/schedule.jsonl"  # which tier ran on which GPU and when (for the validation log)
utc = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
lock = threading.Lock()


def worker(gpu, tiers):
    for tier in tiers:
        command = [sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "validate",
                   "--checkpoint", CHECKPOINT, "--state-sha256", STATE_SHA256, "--choice", CHOICE,
                   "--choice-sha256", CHOICE_SHA256, "--tier", str(tier), "--out", f"{OUT}/tier{tier}"]

        with open(f"{OUT}/logs/tier{tier}.log", "a") as log:
            started, began = time.time(), utc()
            EXITS[tier] = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                         env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}).returncode

        with lock, open(SCHEDULE, "a") as schedule:
            schedule.write(json.dumps({"checkpoint": f"checkpoint-{CHOSEN:04d}", "tier": tier, "gpu": gpu,
                                       "gpu_name": GPUS[gpu], "started": began, "finished": utc(),
                                       "exit": EXITS[tier], "platform": PLATFORM}) + "\n")

        print(f"tier {tier} on GPU {gpu}: exit {EXITS[tier]} after {(time.time() - started) / 3600:.2f} h", flush=True)


threads = [threading.Thread(target=worker, args=(gpu, pending[gpu::len(GPUS)])) for gpu in range(len(GPUS))]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

for tier in TIERS:
    path = f"{OUT}/tier{tier}/validation-tier{tier}.json"

    if os.path.exists(path):
        result = json.load(open(path))
        accuracy, wrong, monitoring = result["decision_accuracy"], result["wrong_interventions_on_healthy"], result["monitoring"]
        print(f"tier {tier}: {result['cases']} cases, decision accuracy {accuracy['k']}/{accuracy['n']}, "
              f"wrong interventions {wrong['k']}/{wrong['n']}, insufficient_evidence precision "
              f"{monitoring['insufficient_evidence_precision']['k']}/{monitoring['insufficient_evidence_precision']['n']}, "
              f"abstain recall {monitoring['abstain_recall']['k']}/{monitoring['abstain_recall']['n']}, "
              f"calls {result['calls']['total']} (non-finite {result['calls']['non_finite']})")
    else:
        print(f"tier {tier}: not complete (exit {EXITS.get(tier)}); see logs/tier{tier}.log")

## Outputs

In [ ]:
archive = ("/kaggle/working" if PLATFORM == "kaggle" else DRIVE) + "/qlora-validation.tar.gz"

with tarfile.open(archive, "w:gz") as results:
    results.add(OUT, arcname="qlora-validation")

print("archive", archive, hashlib.sha256(open(archive, "rb").read()).hexdigest())

if any(code != 0 for code in EXITS.values()):
    raise SystemExit(f"a validation run stopped: {EXITS}; log it before re-running")